## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-04-02-52-16 +0000,nypost,NJ Rep. Tom Kean Jr. loses to Rebecca Bennett ...,https://nypost.com/2026/11/03/us-news/nj-rep-t...
1,2026-11-04-02-51-00 +0000,wsj,"U.S. Stocks, Dollar Set to Weaken as Blue Wave...",https://www.wsj.com/finance/stocks/nikkei-flat...
2,2026-11-04-02-50-48 +0000,chitri,Illinois 8th Congressional District: Melissa B...,https://www.chicagotribune.com/2026/11/03/illi...
3,2026-11-04-02-49-39 +0000,nypost,Dem Keisha Bottoms wins historic Georgia gover...,https://nypost.com/2026/11/03/us-news/dem-keis...
4,2026-11-04-02-47-37 +0000,bbc,US voters name economy and state of democracy ...,https://www.bbc.co.uk/news/articles/c39wz0970e...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2419/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
85,election,46
139,trump,31
48,senate,26
86,day,23
33,race,22


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
140,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...,217
377,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...,163
67,2026-11-04-01-04-38 +0000,chitri,Georgia Senate race: Sen. Jon Ossoff wins reel...,https://www.chicagotribune.com/2026/11/03/geor...,154
128,2026-11-03-23-04-42 +0000,nypost,Dem pollster dishes on which California electi...,https://nypost.com/2026/11/03/us-news/pollster...,152
349,2026-11-03-11-57-00 +0000,wsj,"Follow the WSJ’s Election Day live coverage, a...",https://www.wsj.com/politics/elections/your-ho...,151


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
140,217,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...
67,154,2026-11-04-01-04-38 +0000,chitri,Georgia Senate race: Sen. Jon Ossoff wins reel...,https://www.chicagotribune.com/2026/11/03/geor...
128,152,2026-11-03-23-04-42 +0000,nypost,Dem pollster dishes on which California electi...,https://nypost.com/2026/11/03/us-news/pollster...
123,83,2026-11-03-23-18-22 +0000,nypost,Midterms 2026 elections results map: Live upda...,https://nypost.com/2026/11/03/us-news/midterms...
80,65,2026-11-04-00-55-30 +0000,chitri,North Carolina Senate race: Democrat Roy Coope...,https://www.chicagotribune.com/2026/11/03/nort...
57,61,2026-11-04-01-21-31 +0000,nypost,Chris Pappas halts Sununu comeback in New Hamp...,https://nypost.com/2026/11/03/us-news/chris-pa...
377,56,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...
114,54,2026-11-03-23-39-55 +0000,nyt,OpenAI and Anthropic Set Up ‘War Rooms’ to Fig...,https://www.nytimes.com/2026/11/03/business/ai...
36,52,2026-11-04-02-06-00 +0000,wsj,Democrats jumped out to a strong start Tuesday...,https://www.wsj.com/politics/policy/democrats-...
121,48,2026-11-03-23-21-29 +0000,cbc,AI is making it harder for voters to get accur...,https://www.cbc.ca/news/politics/elections-can...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
